<a href="https://colab.research.google.com/github/18217265596/sx/blob/master/LigandMPNN_Colab_Complete_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LigandMPNN → extract.py → ColabFold 生产版

完整流程：

1. 安装 LigandMPNN 并下载全部官方权重。
2. 上传复合物 PDB，运行 ProteinMPNN / LigandMPNN 等序列设计。
3. 用 `extract.py` 按 `overall_confidence` 提取高分唯一序列。
4. 用户单独指定哪一条链是 de novo 链。
5. de novo 链不搜索 MSA；其余链按序列去重后调用 ColabFold 公共 MMseqs2 MSA server。
6. 若某条非 de novo 链在全部候选中完全相同，只搜索一次 MSA并供所有候选复用；若不同，则对每个唯一序列分别搜索。
7. 使用 AlphaFold2-Multimer v3、3 个模型、无 Amber relaxation，汇总每个模型的 pTM 与 ipTM。

In [ ]:
# 0. 检查运行时
import sys
import platform
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("建议在“运行时 → 更改运行时类型”中选择 T4 GPU。")

In [ ]:
# 1. 克隆 LigandMPNN 并安装兼容依赖
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys

ROOT = Path("/content/LigandMPNN")
RESET_REPOSITORY = True

if RESET_REPOSITORY and ROOT.exists():
    shutil.rmtree(ROOT)

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "https://github.com/dauparas/LigandMPNN.git",
        str(ROOT),
    ],
    check=True,
)

# 保留 Colab 自带 PyTorch/CUDA，不安装官方 requirements.txt 中锁定的旧环境。
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "ProDy==2.6.1",
        "biopython>=1.81",
        "ml-collections==0.1.1",
        "dm-tree==0.1.8",
    ],
    check=True,
)

print("Repository:", ROOT)
subprocess.run(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"],
    check=True,
)

In [ ]:
# 2. 下载 LigandMPNN 官方 get_model_params.sh 当前启用的全部 15 个权重
MODEL_DIR = ROOT / "model_params"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["bash", str(ROOT / "get_model_params.sh"), str(MODEL_DIR)],
    check=True,
)

weight_files = sorted(MODEL_DIR.glob("*.pt"))
if len(weight_files) != 15:
    raise RuntimeError(
        f"预期下载 15 个权重，实际得到 {len(weight_files)} 个。"
    )

print("Downloaded checkpoints:")
for path in weight_files:
    if path.stat().st_size < 1024**2:
        raise RuntimeError(f"权重文件疑似不完整：{path}")
    print(f"  {path.name}: {path.stat().st_size / 1024**2:.1f} MiB")

In [ ]:
# 3. 修复新版 PyTorch 和 NumPy/OpenFold 兼容性，并获取 extract.py
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

run_py = ROOT / "run.py"
source = run_py.read_text(encoding="utf-8")
source = source.replace(
    "torch.load(checkpoint_path, map_location=device)",
    "torch.load(checkpoint_path, map_location=device, weights_only=False)",
)
source = source.replace(
    "torch.load(args.checkpoint_path_sc, map_location=device)",
    "torch.load(args.checkpoint_path_sc, map_location=device, weights_only=False)",
)
run_py.write_text(source, encoding="utf-8")

numpy_aliases = {
    r"\bnp\.int\b": "int",
    r"\bnp\.float\b": "float",
    r"\bnp\.bool\b": "bool",
    r"\bnp\.object\b": "object",
    r"\bnp\.str\b": "str",
    r"\bnp\.complex\b": "complex",
}

changed = []
for py_file in ROOT.rglob("*.py"):
    text = py_file.read_text(encoding="utf-8")
    patched = text
    for pattern, replacement in numpy_aliases.items():
        patched = re.sub(pattern, replacement, patched)
    if patched != text:
        py_file.write_text(patched, encoding="utf-8")
        changed.append(py_file.relative_to(ROOT))

EXTRACT_PY = ROOT / "extract.py"
subprocess.run(
    [
        "wget", "-q",
        "https://raw.githubusercontent.com/18217265596/sx/master/extract.py",
        "-O", str(EXTRACT_PY),
    ],
    check=True,
)

if not EXTRACT_PY.exists() or EXTRACT_PY.stat().st_size == 0:
    raise RuntimeError("extract.py 下载失败。")

print("Compatibility patch completed.")
for path in changed:
    print(" ", path)
print("extract.py:", EXTRACT_PY)

In [ ]:
# 4. 上传一个正式生产用 PDB
from google.colab import files

uploaded = files.upload()
pdb_items = [
    (name, data)
    for name, data in uploaded.items()
    if name.lower().endswith(".pdb")
]

if len(pdb_items) != 1:
    raise ValueError("请一次只上传一个 .pdb 文件。")

name, data = pdb_items[0]
INPUT_DIR = ROOT / "user_inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
USER_PDB = INPUT_DIR / Path(name).name
USER_PDB.write_bytes(data)

print("PDB:", USER_PDB)

## 所有常用参数

下一个代码 cell 是主要参数区。LigandMPNN、`extract.py` 和 ColabFold 的常用参数都集中在这里。

`query_sequence` 和 `jobname` 不需要用户填写：它们会由提取后的完整复合物序列和候选编号自动生成。

In [ ]:
# 5. 所有常用参数（主要只修改本代码块）

# ============================================================
# A. LigandMPNN 权重编号：全部 15 个 .pt
# ============================================================
CHECKPOINT_OPTIONS = {
    1:  ("protein_mpnn", "proteinmpnn_v_48_002.pt", "ProteinMPNN, 0.02 Å noise"),
    2:  ("protein_mpnn", "proteinmpnn_v_48_010.pt", "ProteinMPNN, 0.10 Å noise"),
    3:  ("protein_mpnn", "proteinmpnn_v_48_020.pt", "ProteinMPNN, 0.20 Å noise"),
    4:  ("protein_mpnn", "proteinmpnn_v_48_030.pt", "ProteinMPNN, 0.30 Å noise"),
    5:  ("ligand_mpnn", "ligandmpnn_v_32_005_25.pt", "LigandMPNN, 0.05 Å noise, 25 ligand atoms"),
    6:  ("ligand_mpnn", "ligandmpnn_v_32_010_25.pt", "LigandMPNN, 0.10 Å noise, 25 ligand atoms"),
    7:  ("ligand_mpnn", "ligandmpnn_v_32_020_25.pt", "LigandMPNN, 0.20 Å noise, 25 ligand atoms"),
    8:  ("ligand_mpnn", "ligandmpnn_v_32_030_25.pt", "LigandMPNN, 0.30 Å noise, 25 ligand atoms"),
    9:  ("per_residue_label_membrane_mpnn", "per_residue_label_membrane_mpnn_v_48_020.pt", "MembraneMPNN, per-residue labels"),
    10: ("global_label_membrane_mpnn", "global_label_membrane_mpnn_v_48_020.pt", "MembraneMPNN, global label"),
    11: ("soluble_mpnn", "solublempnn_v_48_002.pt", "SolubleMPNN, 0.02 Å noise"),
    12: ("soluble_mpnn", "solublempnn_v_48_010.pt", "SolubleMPNN, 0.10 Å noise"),
    13: ("soluble_mpnn", "solublempnn_v_48_020.pt", "SolubleMPNN, 0.20 Å noise"),
    14: ("soluble_mpnn", "solublempnn_v_48_030.pt", "SolubleMPNN, 0.30 Å noise"),
    15: ("sidechain_packer", "ligandmpnn_sc_v_32_002_16.pt", "侧链打包权重；不能作为主任务"),
}

print("可用权重编号：")
for number, (_, filename, description) in CHECKPOINT_OPTIONS.items():
    print(f"{number:>2}: {filename:<48} | {description}")

# ---------- LigandMPNN 主任务 ----------
TASK_CHECKPOINT_ID = 6
CHAINS_TO_DESIGN = "A"

SEED = 112
TEMPERATURE = 0.10
BATCH_SIZE = 10
NUMBER_OF_BATCHES = 10
PARSE_ATOMS_WITH_ZERO_OCCUPANCY = 1
SAVE_STATS = 1
FIXED_RESIDUES = ""
REDESIGNED_RESIDUES = ""
VERBOSE = 1

# ---------- 可选侧链打包 ----------
PACK_SIDE_CHAINS = False
SIDECHAIN_CHECKPOINT_ID = 15
NUMBER_OF_PACKS_PER_DESIGN = 1

# ---------- extract.py ----------
EXTRACT_SOURCE_GLOB = "*.fa"
EXTRACT_TOP_N = 20
EXTRACT_COMBINED_FASTA_NAME = "all_sequences.fa"
EXTRACT_TSV_NAME = "top_unique_sequences.tsv"
EXTRACT_FASTA_NAME = "top_unique_sequences.fa"

# ============================================================
# B. ColabFold / AlphaFold2-Multimer
# query_sequence 和 jobname 由前流程自动生成
# ============================================================
RUN_COLABFOLD = True

# 使用公共 ColabFold MMseqs2 MSA server。
COLABFOLD_MSA_SERVER = "https://api.colabfold.com"
COLABFOLD_USE_ENV = True
COLABFOLD_USE_FILTER = True

# de novo 链使用单序列；非 de novo 链使用未配对 MSA。
COLABFOLD_PAIR_MODE = "unpaired"

# 固定使用 AlphaFold2-Multimer v3。
COLABFOLD_MODEL_TYPE = "alphafold2_multimer_v3"

# 用户可修改：建议批量初筛使用 3。
COLABFOLD_NUM_RECYCLES = 3

# 每个复合物只运行 3 个模型。
COLABFOLD_NUM_MODELS = 3
COLABFOLD_MODEL_ORDER = [1, 2, 3]

COLABFOLD_NUM_SEEDS = 1
COLABFOLD_USE_DROPOUT = False
COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE = "auto"
COLABFOLD_MAX_MSA = "auto"

# 不做 Amber relaxation；只关注原始预测的 pTM 和 ipTM。
COLABFOLD_NUM_RELAX = 0

# 不计算 pairwise ipTM / actifpTM 等额外界面评分。
COLABFOLD_CALC_EXTRA_PTM = False

# None 表示运行 extract.py 输出的全部候选。
COLABFOLD_MAX_BINDERS = None

COLABFOLD_KEEP_EXISTING_RESULTS = True
COLABFOLD_JOB_PREFIX = "binder_complex"
COLABFOLD_OUTPUT_DIR_NAME = "colabfold_results"
COLABFOLD_MSA_CACHE_DIR_NAME = "colabfold_msa_cache"
COLABFOLD_A3M_DIR_NAME = "colabfold_complex_a3m"
COLABFOLD_ALL_SCORES_CSV_NAME = "colabfold_all_model_scores.csv"
COLABFOLD_BEST_SCORES_CSV_NAME = "colabfold_best_scores.csv"

# ---------- 最终输出 ----------
DOWNLOAD_RESULTS_ZIP = True
RESULT_ZIP_NAME = "ligandmpnn_colabfold_results"

# ---------- 参数解析与检查：以下一般无需修改 ----------
if TASK_CHECKPOINT_ID not in range(1, 15):
    raise ValueError("TASK_CHECKPOINT_ID 必须为 1–14；15 仅用于侧链打包。")

MODEL_TYPE, CHECKPOINT_NAME, TASK_DESCRIPTION = CHECKPOINT_OPTIONS[TASK_CHECKPOINT_ID]
CHECKPOINT_PATH = MODEL_DIR / CHECKPOINT_NAME

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"未找到主任务权重：{CHECKPOINT_PATH}")

if FIXED_RESIDUES.strip() and REDESIGNED_RESIDUES.strip():
    raise ValueError("FIXED_RESIDUES 与 REDESIGNED_RESIDUES 不能同时使用。")

if EXTRACT_TOP_N < 1:
    raise ValueError("EXTRACT_TOP_N 必须至少为 1。")

if COLABFOLD_NUM_MODELS != 3 or COLABFOLD_MODEL_ORDER != [1, 2, 3]:
    raise ValueError("当前生产流程固定使用模型 1、2、3，共 3 个模型。")

if COLABFOLD_NUM_RELAX != 0:
    raise ValueError("当前流程不进行 Amber relaxation，请保持 COLABFOLD_NUM_RELAX = 0。")

if COLABFOLD_CALC_EXTRA_PTM:
    raise ValueError("当前流程不计算额外界面评分，请保持 COLABFOLD_CALC_EXTRA_PTM = False。")

if PACK_SIDE_CHAINS:
    if SIDECHAIN_CHECKPOINT_ID != 15:
        raise ValueError("当前侧链打包权重编号必须为 15。")
    _, SIDECHAIN_CHECKPOINT_NAME, _ = CHECKPOINT_OPTIONS[SIDECHAIN_CHECKPOINT_ID]
    SIDECHAIN_CHECKPOINT_PATH = MODEL_DIR / SIDECHAIN_CHECKPOINT_NAME
    if not SIDECHAIN_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"未找到侧链打包权重：{SIDECHAIN_CHECKPOINT_PATH}")
else:
    SIDECHAIN_CHECKPOINT_PATH = None

print("\n当前任务：")
print(" Model type:", MODEL_TYPE)
print(" Checkpoint:", CHECKPOINT_PATH)
print(" Description:", TASK_DESCRIPTION)
print(" Chains to design:", CHAINS_TO_DESIGN or "all")
print(" LigandMPNN total sequences:", BATCH_SIZE * NUMBER_OF_BATCHES)
print(" extract.py top N:", EXTRACT_TOP_N)
print(" ColabFold enabled:", RUN_COLABFOLD)
print(" ColabFold model:", COLABFOLD_MODEL_TYPE)
print(" ColabFold models:", COLABFOLD_MODEL_ORDER)
print(" ColabFold recycles:", COLABFOLD_NUM_RECYCLES)
print(" ColabFold relax:", COLABFOLD_NUM_RELAX)

## 指定 de novo 链

下面是独立的用户输入 cell。填写 **PDB 中哪一条链是 de novo 设计链**。

该链在 ColabFold 中只使用自身序列，不调用 MSA server。所有其他链都会在 `extract.py` 完成后接受逐链一致性检查。

In [ ]:
# 6. 用户指定 de novo 链（独立参数 cell）
COLABFOLD_DE_NOVO_CHAIN = "A"

COLABFOLD_DE_NOVO_CHAIN = COLABFOLD_DE_NOVO_CHAIN.strip()
if not COLABFOLD_DE_NOVO_CHAIN:
    raise ValueError("必须指定 COLABFOLD_DE_NOVO_CHAIN。")
if "," in COLABFOLD_DE_NOVO_CHAIN:
    raise ValueError("当前流程一次只接受一条 de novo 链，例如 'A'。")

print("de novo chain:", COLABFOLD_DE_NOVO_CHAIN)
print("该链不会调用 MSA server。")

In [ ]:
# 7. 定义完整日志运行函数
from typing import Sequence

def run_and_show(command: Sequence[str], cwd: Path = ROOT):
    env = os.environ.copy()
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    env["PYTHONUNBUFFERED"] = "1"

    print("Running command:\n")
    print(" ".join(map(str, command)))
    print("\n" + "=" * 90)

    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout)
    print("=" * 90)
    print("Exit status:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError("命令运行失败，完整报错见上方。")

    return result

In [ ]:
# 8. 运行 LigandMPNN 正式生产任务
USER_OUT = ROOT / "outputs" / USER_PDB.stem
shutil.rmtree(USER_OUT, ignore_errors=True)

checkpoint_flags = {
    "protein_mpnn": "--checkpoint_protein_mpnn",
    "ligand_mpnn": "--checkpoint_ligand_mpnn",
    "soluble_mpnn": "--checkpoint_soluble_mpnn",
    "per_residue_label_membrane_mpnn": "--checkpoint_per_residue_label_membrane_mpnn",
    "global_label_membrane_mpnn": "--checkpoint_global_label_membrane_mpnn",
}

command = [
    sys.executable, "-u", "run.py",
    "--model_type", MODEL_TYPE,
    checkpoint_flags[MODEL_TYPE], str(CHECKPOINT_PATH),
    "--seed", str(SEED),
    "--pdb_path", str(USER_PDB),
    "--out_folder", str(USER_OUT),
    "--batch_size", str(BATCH_SIZE),
    "--number_of_batches", str(NUMBER_OF_BATCHES),
    "--temperature", str(TEMPERATURE),
    "--parse_atoms_with_zero_occupancy", str(PARSE_ATOMS_WITH_ZERO_OCCUPANCY),
    "--save_stats", str(SAVE_STATS),
    "--verbose", str(VERBOSE),
]

if CHAINS_TO_DESIGN.strip():
    command.extend(["--chains_to_design", CHAINS_TO_DESIGN.strip()])

if FIXED_RESIDUES.strip():
    command.extend(["--fixed_residues", FIXED_RESIDUES.strip()])

if REDESIGNED_RESIDUES.strip():
    command.extend(["--redesigned_residues", REDESIGNED_RESIDUES.strip()])

if PACK_SIDE_CHAINS:
    command.extend([
        "--pack_side_chains", "1",
        "--checkpoint_path_sc", str(SIDECHAIN_CHECKPOINT_PATH),
        "--number_of_packs_per_design", str(NUMBER_OF_PACKS_PER_DESIGN),
    ])

run_and_show(command)
print("Production output:", USER_OUT)

In [ ]:
# 9. 调用 extract.py 提取 overall_confidence 最高的唯一序列
SEQ_DIR = USER_OUT / "seqs"
source_fastas = sorted(SEQ_DIR.glob(EXTRACT_SOURCE_GLOB))

if not source_fastas:
    raise FileNotFoundError(
        f"在 {SEQ_DIR} 中没有匹配 {EXTRACT_SOURCE_GLOB!r} 的 FASTA。"
    )

COMBINED_FASTA = USER_OUT / EXTRACT_COMBINED_FASTA_NAME
with COMBINED_FASTA.open("w", encoding="utf-8") as output:
    for fasta in source_fastas:
        text = fasta.read_text(encoding="utf-8")
        output.write(text)
        if text and not text.endswith("\n"):
            output.write("\n")

EXTRACT_TSV = USER_OUT / EXTRACT_TSV_NAME
extract_command = [
    sys.executable, str(EXTRACT_PY),
    "--input", str(COMBINED_FASTA),
    "--top", str(EXTRACT_TOP_N),
]

extract_result = subprocess.run(
    extract_command,
    cwd=str(ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

if extract_result.returncode != 0:
    print(extract_result.stderr)
    raise RuntimeError("extract.py 运行失败。")

EXTRACT_TSV.write_text(extract_result.stdout, encoding="utf-8")

EXTRACT_FASTA = USER_OUT / EXTRACT_FASTA_NAME
with EXTRACT_FASTA.open("w", encoding="utf-8") as handle:
    for line in extract_result.stdout.splitlines():
        if not line.strip():
            continue
        rank, confidence, record_id, sequence = line.split("\t", 3)
        handle.write(
            f">rank={rank}, overall_confidence={confidence}, id={record_id}\n"
            f"{sequence}\n"
        )

print("Combined input:", COMBINED_FASTA)
print("extract.py command:", " ".join(extract_command))
print("TSV output:", EXTRACT_TSV)
print("FASTA output:", EXTRACT_FASTA)
print("\nExtracted records:\n")
print(extract_result.stdout)

In [ ]:
# 10. 安装 ColabFold（与官方 AlphaFold2.ipynb 相同的安装来源）
if RUN_COLABFOLD:
    COLABFOLD_READY = Path("/content/COLABFOLD_READY")

    if not COLABFOLD_READY.exists():
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "--no-warn-conflicts",
                "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold",
            ],
            check=True,
        )
        COLABFOLD_READY.touch()

    import colabfold
    print("ColabFold:", getattr(colabfold, "__version__", "installed"))
else:
    print("RUN_COLABFOLD=False，跳过安装。")

In [ ]:
# 11. 解析候选链，检查一致性，并规划 MSA 复用
from collections import OrderedDict, defaultdict
from dataclasses import dataclass
import csv
import hashlib

@dataclass
class CandidateComplex:
    rank: int
    confidence: float
    record_id: str
    chain_sequences: OrderedDict

def pdb_chain_order(pdb_path: Path):
    order = []
    seen = set()
    with pdb_path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.startswith("ATOM"):
                chain = line[21].strip() or "_"
                if chain not in seen:
                    seen.add(chain)
                    order.append(chain)
    return order

CHAIN_ORDER = pdb_chain_order(USER_PDB)
if not CHAIN_ORDER:
    raise ValueError("输入 PDB 中未识别到蛋白 ATOM 链。")

if COLABFOLD_DE_NOVO_CHAIN not in CHAIN_ORDER:
    raise ValueError(
        f"de novo 链 {COLABFOLD_DE_NOVO_CHAIN!r} 不在 PDB 链顺序 {CHAIN_ORDER} 中。"
    )

candidates = []
for line in EXTRACT_TSV.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    rank_s, confidence_s, record_id, full_sequence = line.split("\t", 3)
    parts = full_sequence.split(":")

    if len(parts) != len(CHAIN_ORDER):
        raise ValueError(
            "LigandMPNN FASTA 的链数与输入 PDB 不一致："
            f"rank={rank_s}, FASTA chains={len(parts)}, PDB chains={len(CHAIN_ORDER)}。"
            "请确认 --fasta_seq_separation 保持默认 ':'，并确认 PDB 链顺序未改变。"
        )

    chain_sequences = OrderedDict(zip(CHAIN_ORDER, parts))
    candidates.append(
        CandidateComplex(
            rank=int(rank_s),
            confidence=float(confidence_s),
            record_id=record_id,
            chain_sequences=chain_sequences,
        )
    )

if COLABFOLD_MAX_BINDERS is not None:
    candidates = candidates[: int(COLABFOLD_MAX_BINDERS)]

if not candidates:
    raise ValueError("没有可供 ColabFold 预测的候选序列。")

# 对每条链统计候选中的唯一序列。
chain_unique_sequences = OrderedDict()
for chain in CHAIN_ORDER:
    chain_unique_sequences[chain] = list(
        OrderedDict.fromkeys(
            candidate.chain_sequences[chain]
            for candidate in candidates
        )
    )

MSA_PLAN_CSV = USER_OUT / "colabfold_msa_plan.csv"
with MSA_PLAN_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow([
        "chain", "is_de_novo", "candidate_count",
        "unique_sequence_count", "msa_policy",
    ])

    print("Chain-wise MSA plan:")
    for chain, unique_sequences in chain_unique_sequences.items():
        is_de_novo = chain == COLABFOLD_DE_NOVO_CHAIN

        if is_de_novo:
            policy = "single_sequence_no_msa"
        elif len(unique_sequences) == 1:
            policy = "one_shared_msa_for_all_candidates"
        else:
            policy = "one_msa_per_unique_sequence"

        writer.writerow([
            chain,
            is_de_novo,
            len(candidates),
            len(unique_sequences),
            policy,
        ])

        print(
            f"  chain {chain}: de_novo={is_de_novo}, "
            f"unique={len(unique_sequences)}, policy={policy}"
        )

print("MSA plan:", MSA_PLAN_CSV)

In [ ]:
# 12. 调用公共 MMseqs2 MSA server；de novo 链不搜索
if RUN_COLABFOLD:
    from colabfold.colabfold import run_mmseqs2
    from colabfold.input import msa_to_str

    MSA_CACHE_DIR = USER_OUT / COLABFOLD_MSA_CACHE_DIR_NAME
    COMPLEX_A3M_DIR = USER_OUT / COLABFOLD_A3M_DIR_NAME
    MSA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    COMPLEX_A3M_DIR.mkdir(parents=True, exist_ok=True)

    def single_sequence_a3m(sequence: str, name: str = "query") -> str:
        return f">{name}\n{sequence}\n"

    def msa_cache_key(sequence: str) -> str:
        return hashlib.sha1(sequence.encode("utf-8")).hexdigest()[:16]

    msa_cache = {}

    # 非 de novo 链：每个唯一序列只查询一次。
    for chain, unique_sequences in chain_unique_sequences.items():
        if chain == COLABFOLD_DE_NOVO_CHAIN:
            continue

        for sequence in unique_sequences:
            key = msa_cache_key(sequence)
            cache_file = MSA_CACHE_DIR / f"{chain}_{key}.a3m"

            if cache_file.exists() and cache_file.stat().st_size > 0:
                a3m_text = cache_file.read_text(encoding="utf-8")
                print(f"Reuse cached MSA: chain={chain}, key={key}")
            else:
                print(f"Query MSA server: chain={chain}, key={key}, length={len(sequence)}")
                prefix = str(MSA_CACHE_DIR / f"mmseqs_{chain}_{key}")
                result = run_mmseqs2(
                    sequence,
                    prefix,
                    use_env=COLABFOLD_USE_ENV,
                    use_filter=COLABFOLD_USE_FILTER,
                    use_templates=False,
                    use_pairing=False,
                    host_url=COLABFOLD_MSA_SERVER,
                    user_agent="ligandmpnn-colabfold-workflow/1.0",
                )
                a3m_text = result[0]
                cache_file.write_text(a3m_text, encoding="utf-8")

            msa_cache[(chain, sequence)] = a3m_text

    # 为每个候选构造 custom complex A3M。
    # de novo 链只有 query；其余链使用缓存的 unpaired MSA。
    complex_manifest = []
    for candidate in candidates:
        chain_sequences = list(candidate.chain_sequences.values())
        unpaired_msas = []

        for chain, sequence in candidate.chain_sequences.items():
            if chain == COLABFOLD_DE_NOVO_CHAIN:
                unpaired_msas.append(
                    single_sequence_a3m(sequence, f"de_novo_chain_{chain}")
                )
            else:
                unpaired_msas.append(msa_cache[(chain, sequence)])

        # 当前流程按 PDB 链顺序保留每条链。
        # 若完全相同的序列同时出现在不同链且需要不同 MSA 策略，会造成歧义。
        for i, chain_i in enumerate(CHAIN_ORDER):
            for j, chain_j in enumerate(CHAIN_ORDER):
                if i >= j:
                    continue
                seq_i = candidate.chain_sequences[chain_i]
                seq_j = candidate.chain_sequences[chain_j]
                if seq_i == seq_j and (
                    (chain_i == COLABFOLD_DE_NOVO_CHAIN)
                    != (chain_j == COLABFOLD_DE_NOVO_CHAIN)
                ):
                    raise ValueError(
                        "同一候选中存在完全相同序列，但一条被标为 de novo、另一条不是。"
                        "ColabFold 会把完全相同序列视作同聚体，无法对两个拷贝使用不同 MSA 策略。"
                    )

        # 折叠完全相同的链序列为 cardinality。
        unique_sequences = []
        cardinalities = []
        unique_msas = []

        for sequence, a3m_text in zip(chain_sequences, unpaired_msas):
            if sequence in unique_sequences:
                idx = unique_sequences.index(sequence)
                cardinalities[idx] += 1
            else:
                unique_sequences.append(sequence)
                cardinalities.append(1)
                unique_msas.append(a3m_text)

        complex_a3m = msa_to_str(
            unpaired_msa=unique_msas,
            paired_msa=None,
            query_seqs_unique=unique_sequences,
            query_seqs_cardinality=cardinalities,
        )

        jobname = (
            f"{COLABFOLD_JOB_PREFIX}_"
            f"{candidate.rank:03d}_id_{candidate.record_id}"
        )
        a3m_path = COMPLEX_A3M_DIR / f"{jobname}.a3m"
        a3m_path.write_text(complex_a3m, encoding="utf-8")

        complex_manifest.append({
            "jobname": jobname,
            "rank": candidate.rank,
            "overall_confidence": candidate.confidence,
            "record_id": candidate.record_id,
            "query_sequence": ":".join(chain_sequences),
            "a3m_path": str(a3m_path),
        })

    MANIFEST_CSV = USER_OUT / "colabfold_complex_manifest.csv"
    with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=complex_manifest[0].keys())
        writer.writeheader()
        writer.writerows(complex_manifest)

    print("Complex A3M directory:", COMPLEX_A3M_DIR)
    print("Complex manifest:", MANIFEST_CSV)
    print("Prepared complexes:", len(complex_manifest))
else:
    print("RUN_COLABFOLD=False，跳过 MSA。")

In [ ]:
# 13. 运行 AlphaFold2-Multimer v3：3 个模型、无 relaxation
if RUN_COLABFOLD:
    from colabfold.download import download_alphafold_params
    from colabfold.utils import setup_logging
    from colabfold.batch import get_queries, run, set_model_type

    COLABFOLD_RESULT_DIR = USER_OUT / COLABFOLD_OUTPUT_DIR_NAME
    COLABFOLD_RESULT_DIR.mkdir(parents=True, exist_ok=True)

    queries, is_complex = get_queries(COMPLEX_A3M_DIR)
    if not is_complex:
        raise RuntimeError("生成的 A3M 未被 ColabFold 识别为复合物。")

    resolved_model_type = set_model_type(
        is_complex,
        COLABFOLD_MODEL_TYPE,
    )

    DATA_DIR = Path("/content/colabfold_params")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    download_alphafold_params(resolved_model_type, DATA_DIR)

    if COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE == "auto":
        recycle_tolerance = 0.5
    else:
        recycle_tolerance = float(
            COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE
        )

    if COLABFOLD_MAX_MSA == "auto":
        max_msa = None
    else:
        max_msa = COLABFOLD_MAX_MSA

    # 与官方 notebook 一致：multimer 且显式限制 max_msa 时关闭 cluster profile。
    if "multimer" in resolved_model_type and max_msa is not None:
        use_cluster_profile = False
    else:
        use_cluster_profile = True

    setup_logging(COLABFOLD_RESULT_DIR / "log.txt")

    run(
        queries=queries,
        result_dir=COLABFOLD_RESULT_DIR,
        use_templates=False,
        custom_template_path=None,
        num_relax=COLABFOLD_NUM_RELAX,
        msa_mode="custom",
        model_type=resolved_model_type,
        num_models=COLABFOLD_NUM_MODELS,
        num_recycles=int(COLABFOLD_NUM_RECYCLES),
        relax_max_iterations=0,
        recycle_early_stop_tolerance=recycle_tolerance,
        num_seeds=COLABFOLD_NUM_SEEDS,
        use_dropout=COLABFOLD_USE_DROPOUT,
        model_order=COLABFOLD_MODEL_ORDER,
        is_complex=True,
        data_dir=DATA_DIR,
        keep_existing_results=COLABFOLD_KEEP_EXISTING_RESULTS,
        rank_by="auto",
        pair_mode=COLABFOLD_PAIR_MODE,
        pairing_strategy="greedy",
        stop_at_score=100,
        prediction_callback=None,
        dpi=200,
        zip_results=False,
        save_all=False,
        max_msa=max_msa,
        use_cluster_profile=use_cluster_profile,
        input_features_callback=None,
        save_recycles=False,
        user_agent="ligandmpnn-colabfold-workflow/1.0",
        calc_extra_ptm=COLABFOLD_CALC_EXTRA_PTM,
    )

    print("ColabFold output:", COLABFOLD_RESULT_DIR)
else:
    print("RUN_COLABFOLD=False，跳过结构预测。")

In [ ]:
# 14. 汇总所有模型及每个候选第一名的 pTM / ipTM
if RUN_COLABFOLD:
    import json
    import pandas as pd
    from IPython.display import display

    manifest_by_job = {
        row["jobname"]: row
        for row in complex_manifest
    }

    rows = []
    score_files = sorted(
        COLABFOLD_RESULT_DIR.glob("*_scores_*.json")
    )

    if not score_files:
        raise FileNotFoundError(
            f"在 {COLABFOLD_RESULT_DIR} 中未找到 *_scores_*.json。"
        )

    pattern = re.compile(
        r"^(?P<job>.+)_scores_rank_(?P<rank>\d+)_"
        r"(?P<tag>.+)\.json$"
    )

    for score_file in score_files:
        match = pattern.match(score_file.name)
        if not match:
            continue

        jobname_from_file = match.group("job")
        jobname = jobname_from_file.removesuffix(".custom")
        prediction_rank = int(match.group("rank"))
        scores = json.loads(score_file.read_text(encoding="utf-8"))
        manifest = manifest_by_job.get(jobname, {})

        rows.append({
            "jobname": jobname,
            "ligandmpnn_rank": manifest.get("rank"),
            "ligandmpnn_overall_confidence": manifest.get("overall_confidence"),
            "record_id": manifest.get("record_id"),
            "prediction_rank": prediction_rank,
            "model_tag": match.group("tag"),
            "ptm": scores.get("ptm"),
            "iptm": scores.get("iptm"),
            "ranking_confidence": scores.get("ranking_confidence"),
            "score_json": str(score_file),
        })

    if not rows:
        raise RuntimeError("找到了 score JSON，但文件名无法解析。")

    all_scores = pd.DataFrame(rows).sort_values(
        ["ligandmpnn_rank", "prediction_rank"]
    )

    best_scores = (
        all_scores.sort_values(["jobname", "prediction_rank"])
        .groupby("jobname", as_index=False)
        .first()
        .sort_values(
            ["iptm", "ptm"],
            ascending=[False, False],
            na_position="last",
        )
    )

    ALL_SCORES_CSV = USER_OUT / COLABFOLD_ALL_SCORES_CSV_NAME
    BEST_SCORES_CSV = USER_OUT / COLABFOLD_BEST_SCORES_CSV_NAME

    all_scores.to_csv(ALL_SCORES_CSV, index=False)
    best_scores.to_csv(BEST_SCORES_CSV, index=False)

    print("All model scores:", ALL_SCORES_CSV)
    print("Best score per candidate:", BEST_SCORES_CSV)
    display(best_scores[[
        "jobname",
        "ligandmpnn_rank",
        "ligandmpnn_overall_confidence",
        "prediction_rank",
        "ptm",
        "iptm",
    ]])
else:
    print("RUN_COLABFOLD=False，无评分可汇总。")

In [ ]:
# 15. 打包并下载 LigandMPNN、extract.py、MSA 与 ColabFold 全部结果
from google.colab import files

archive_path = shutil.make_archive(
    f"/content/{RESULT_ZIP_NAME}",
    "zip",
    root_dir=str(USER_OUT),
)

print("Archive:", archive_path)

if DOWNLOAD_RESULTS_ZIP:
    files.download(archive_path)